In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import datasets, transforms, models
import numpy as np
from sklearn.metrics import classification_report
from sklearn.metrics import classification_report
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

# Этап 1. Загрузка данных

DATA_DIR = 'dataset'
BATCH_SIZE = 32
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, 'train'),
    transform=train_transform
)

class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Найдено классов: {num_classes}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

test_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, 'test'),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Этап 2. Создание модели

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)
print("Модель готова")

# Этап 3. Обучение

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    train_losses.append(running_loss / len(train_loader))
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * correct / total
    val_accuracies.append(val_acc)
    
    print(f'Эпоха {epoch+1}: Loss={train_losses[-1]:.4f}, Val Acc={val_acc:.2f}%')

torch.save(model.state_dict(), 'meds_classifier.pt')
print("Модель сохранена")

# Этап 4. Оценка качества

model.eval()
val_preds = []
val_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = outputs.max(1)
        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.numpy())

val_accuracy = np.sum(np.array(val_preds) == np.array(val_labels)) / len(val_labels)
print(f'\nТочность на валидации: {val_accuracy:.2%}')

if val_accuracy > 0.75:
    print("Целевая точность выше 75% на валидации")
else:
    print(f"Точность на валидации {val_accuracy:.2%} ниже целевых 75%")

print('Метрики на валидации')

class_names = full_dataset.classes
report_dict = classification_report(
    val_labels, 
    val_preds, 
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

df_val = pd.DataFrame(report_dict).transpose()
df_val = df_val[~df_val.index.isin(['accuracy', 'macro avg', 'weighted avg'])]
print(df_val[['precision', 'recall', 'f1-score', 'support']].to_string())
print('=' * 120)

test_preds = []
test_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = outputs.max(1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.numpy())

test_accuracy = np.sum(np.array(test_preds) == np.array(test_labels)) / len(test_labels)
print(f'\nТочность на тесте: {test_accuracy:.2%}')

print(f'Валидация: {val_accuracy:.2%}')
print(f'Тест:      {test_accuracy:.2%}')

print('Метрики на тесте')

report_dict_test = classification_report(
    test_labels, 
    test_preds, 
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

df_test = pd.DataFrame(report_dict_test).transpose()
df_test = df_test[~df_test.index.isin(['accuracy', 'macro avg', 'weighted avg'])]
print(df_test[['precision', 'recall', 'f1-score', 'support']].to_string())
print('=' * 120)


df_test_sorted = df_test.sort_values('f1-score', ascending=False)

print('\n5 классов с лучшим F1-SCORE на тестовых данных:')
for i, (idx, row) in enumerate(df_test_sorted.head(5).iterrows(), 1):
    print(f"   {i}. {idx[:40]:<40} F1={row['f1-score']:.3f} (примеров: {row['support']:.0f})")

print('\n5 классов с худшим F1-SCORE на тестовых данных:')
for i, (idx, row) in enumerate(df_test_sorted.tail(5).iterrows(), 1):
    print(f"   {i}. {idx[:40]:<40} F1={row['f1-score']:.3f} (примеров: {row['support']:.0f})")


worst_classes = df_test_sorted.tail(5)
print('\n1. На каких 5 классах модель ошибается чаще всего?')
for i, (idx, row) in enumerate(worst_classes.iterrows(), 1):
    error_rate = 1 - row['f1-score']
    print(f"   {i}. {idx[:40]:<40} - ошибок: ~{error_rate:.2%}")

print('\n2. Почему модель может ошибаться на этих классах?')
print('   • Мало обучающих примеров:')
small_support = worst_classes[worst_classes['support'] < 20]
for idx, row in small_support.iterrows():
    print(f'     - {idx[:30]} (всего {row["support"]:.0f} примеров)')
print('   • Визуальное сходство с другими классами')
print('   • Разные ракурсы и освещение на фото')

best_classes = df_test_sorted.head(5)
print('\n3. На каких классах модель ошибается меньше всего?')
for i, (idx, row) in enumerate(best_classes.iterrows(), 1):
    error_rate = 1 - row['f1-score']
    print(f"   {i}. {idx[:40]:<40} - ошибок: {error_rate:.2%}")

print('\n4. Почему эти классы распознаются хорошо?')
print('   • Достаточно обучающих примеров:')
for idx, row in best_classes.head(3).iterrows():
    print(f'     - {idx[:30]} ({row["support"]:.0f} примеров)')
print('   • Уникальные визуальные признаки')
print('   • Четкие фото с минимальными вариациями')

print('\n5. Как можно улучшить точность классификатора?')
print('   • Добавить больше аугментации данных')
print('   • Увеличить число эпох обучения')
print('   • Разморозить больше слоев для тонкой настройки')
print('   • Добавить примеров для проблемных классов:')
for idx, row in worst_classes.head(3).iterrows():
    print(f'     - {idx[:30]} (сейчас {row["support"]:.0f} примеров)')

print('\n6. Как ещё можно проанализировать результаты?')
print('   • Построить матрицу ошибок')
print('   • Проанализировать неправильно классифицированные изображения')

Устройство: cpu
Найдено классов: 84
Модель готова
Эпоха 1: Loss=4.1620, Val Acc=17.62%
Эпоха 2: Loss=3.0025, Val Acc=43.10%
Эпоха 3: Loss=2.2653, Val Acc=51.17%
Эпоха 4: Loss=1.7918, Val Acc=58.39%
Эпоха 5: Loss=1.4387, Val Acc=66.45%
Эпоха 6: Loss=1.2060, Val Acc=70.06%
Эпоха 7: Loss=1.0043, Val Acc=72.61%
Эпоха 8: Loss=0.8930, Val Acc=77.49%
Эпоха 9: Loss=0.7855, Val Acc=77.49%
Эпоха 10: Loss=0.6927, Val Acc=79.41%
Модель сохранена

Точность на валидации: 79.41%
Целевая точность выше 75% на валидации
Метрики на валидации
                                  precision    recall  f1-score  support
acc_long_600_mg                    1.000000  1.000000  1.000000      5.0
advil_ultra_forte                  1.000000  0.857143  0.923077      7.0
akineton_2_mg                      0.875000  1.000000  0.933333      7.0
algoflex_forte_dolo_400_mg         1.000000  1.000000  1.000000      4.0
algoflex_rapid_400_mg              0.800000  1.000000  0.888889      4.0
algopyrin_500_mg                 